# Supplementary Figure 10

**Mean absolute percentage difference between the manually measured maximum axis length and the major
axis length calculated in Mycol**

- **Does** — per-class mean of `|manual - mycol| / mycol` over the 47 hand-measured larvae, SD error
  bars, every larva overlaid as a point
- **Style** — matches the Figure 4 panels: 4.5 in at 300 dpi, type calibrated to 19.5 px, classes in
  `#5289C7` / `#4EB265`, seeded jitter
- **Needs** — `assets/manual_vs_mycol_larvae_measurements.csv`, the same file Figure 4 panel d plots
  (a copy lives with each figure so neither depends on the other)
- **Kernel** — `mycol_colonies_env`, run top to bottom


## The measurements


In [ ]:
from pathlib import Path

import matplotlib
matplotlib.use("Agg")   # headless, and keeps the canvas clear of Retina 2x device
                        # scaling, which snaps figure sizes to whole device pixels
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from PIL import Image

OUTPUT = Path("output")
OUTPUT.mkdir(exist_ok=True)

# Figure 4 scales a 4.5 in panel drawn at 300 dpi into a 565.33 px column, and
# sets its type so labels land at 19.5 px there. Same arithmetic here, so this
# figure sets at the same size.
COL_W = (1788 - 2 * 20 - 2 * 26) / 3
TEXT_PX, PANEL_IN, DPI = 19.5, 4.5, 300
FONT_PT = TEXT_PX / ((DPI / 72) * (COL_W / (PANEL_IN * DPI)))
plt.rcParams.update({"font.size": FONT_PT, "axes.labelsize": FONT_PT,
                     "xtick.labelsize": FONT_PT, "ytick.labelsize": FONT_PT})

# normal is clsA, abnormal is clsB in Figure 4's workflow strip
CLASS_COLOURS = {"normal": "#5289C7", "abnormal": "#4EB265"}
CLASS_ORDER = ["normal", "abnormal"]

X_MANUAL, Y_MYCOL = "Manual Length", "Mycol: Major Axis Length"
merged = pd.read_csv("assets/manual_vs_mycol_larvae_measurements.csv")
merged["abs_pct_diff"] = ((merged[X_MANUAL] - merged[Y_MYCOL]) / merged[Y_MYCOL] * 100).abs()
data = [merged.loc[merged["mask label"] == c, "abs_pct_diff"].to_numpy() for c in CLASS_ORDER]

print(f"FONT_PT {FONT_PT:.2f}  ->  {TEXT_PX} px at Figure 4 panel scale")
for cls, d in zip(CLASS_ORDER, data):
    print(f"  {cls:9s} n={len(d):2d}  mean {d.mean():.2f}%  sd {d.std(ddof=1):.2f}"
          f"  range {d.min():.2f}-{d.max():.2f}")
print(f"  overall   n={len(merged):2d}  mean {merged['abs_pct_diff'].mean():.2f}%")


## Draw


In [ ]:
BAR_W = 0.75                        # the width Figure 4e gives its violins
JITTER = 0.25                       # app's jitter, as a fraction of that width
rng = np.random.default_rng(42)     # seeded so the point cloud is reproducible

fig, ax = plt.subplots(figsize=(PANEL_IN, PANEL_IN), layout="constrained")
for i, (cls, d) in enumerate(zip(CLASS_ORDER, data)):
    ax.bar(i, d.mean(), width=BAR_W, color=CLASS_COLOURS[cls], alpha=0.85,
           edgecolor="#0f172a", linewidth=1.2, zorder=2)
    ax.errorbar(i, d.mean(), yerr=d.std(ddof=1), fmt="none", ecolor="#0f172a",
                elinewidth=1.2, capsize=6, capthick=1.2, zorder=4)
    # every larva overlaid as a jittered black point, centred on its bar
    x = i + rng.uniform(-JITTER * BAR_W / 2, JITTER * BAR_W / 2, len(d))
    ax.scatter(x, d, s=5, color="black", alpha=0.45, linewidths=0, zorder=5)

ax.set_xticks(range(len(CLASS_ORDER)),
              [f"{cls.capitalize()}\n(n={len(d)})" for cls, d in zip(CLASS_ORDER, data)])
ax.set_ylabel("Absolute percentage difference (%)", fontsize=FONT_PT)
ax.set_ylim(0, None)
ax.grid(True, axis="y", lw=0.4, color="#ededed")
ax.set_axisbelow(True)
for sp in ("top", "right"):
    ax.spines[sp].set_visible(False)

out = OUTPUT / "Figure_S10.png"
fig.savefig(out, dpi=DPI, facecolor="white")
print(f"wrote {out}  {out.stat().st_size:,} B")

im = Image.open(out)
im.thumbnail((700, 700))
display(im.convert("RGB"))
